In [ ]:
import sqlalchemy as sa

# Create the connection string for your local database with the new password
db_uri = "postgresql+psycopg2://:@localhost:/"

# Create the Engine object, keeping pool_pre_ping for reliability
engine = sa.create_engine(db_uri, pool_pre_ping=True)

print("✅ Connection engine for the local 'layereddb' is ready.")

✅ Connection engine for the local 'layereddb' is ready.


In [8]:
from sqlalchemy import inspect
# Create an inspector object from engine
inspector = inspect(engine)

# Get the list of schema names
schemas = inspector.get_schema_names()

print("Available schemas in the database:")
print(schemas)

Available schemas in the database:
['berlin_labels', 'berlin_recommender', 'berlin_source_data', 'dashboard_data', 'information_schema', 'public']


In [9]:
# Get the list of table names for the 'berlin_source_data' schema
tables_in_schema = inspector.get_table_names(schema='berlin_source_data')

# Print the list of tables
print("\nTables in schema 'berlin_source_data':")
print(tables_in_schema)


Tables in schema 'berlin_source_data':
['theaters', 'pools_refactored', 'theaters_backup_neigh_final', 'night_clubs', 'district_level_aggregated', 'bus_tram_stops', 'malls', 'banks', 'social_clubs_activities', 'veterinary_clinics_martin_svitek', 'hospitals_refactored', 'pharmacies', 'supermarkets', 'bike_lanes', 'pools', 'hospitals', 'land_prices', 'test_table_george_smelin', 'gyms', 'universities', 'venues', 'dental_offices', 'post_offices', 'kindergartens', 'sbahn', 'schools', 'short_term_listings', 'districts', 'ubahn', 'long_term_listings', 'veterinary_clinics', 'milieuschutz_protection_zones', 'neighborhoods', 'parks', 'regional_statistics', 'crime_statistics', 'districts_pop_stat', 'playgrounds', 'rent_stats_per_neighborhood']


In [10]:
import pandas as pd
# Full table name, including the schema
table_name = 'berlin_source_data.venues'

# SQL query to select all data (*) from your table
query = f"SELECT * FROM {table_name}"

# Execute the query and load the result into a Pandas DataFrame
# The read_sql function takes an SQL query and a connection object (engine)
venues = pd.read_sql(query, engine)

print(venues.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8035 entries, 0 to 8034
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   venue_id               8035 non-null   object 
 1   district_id            8035 non-null   object 
 2   name                   8035 non-null   object 
 3   district               8035 non-null   object 
 4   category               8035 non-null   object 
 5   cuisine                4569 non-null   object 
 6   phone                  1993 non-null   object 
 7   address                8033 non-null   object 
 8   latitude               8035 non-null   float64
 9   longitude              8035 non-null   float64
 10  website                2472 non-null   object 
 11  opening_hours_dict     5603 non-null   object 
 12  opening_hours          5603 non-null   object 
 13  postal_code            7531 non-null   object 
 14  neighborhood           8035 non-null   object 
 15  take

In [11]:
import ast
import datetime

# --- Helper Function to check if a specific time falls within a slot ---
def is_time_in_slot(check_time, slot_string):
    """Checks if check_time (datetime.time) is within a slot_string ('HH:MM-HH:MM' or ['HH:MM', 'HH:MM']). Handles cross-midnight."""
    try:
        start_time = None
        end_time = None

        # Format 1: ['HH:MM', 'HH:MM']
        if isinstance(slot_string, list) and len(slot_string) == 2:
             start_str, end_str = slot_string
             if len(start_str)==5 and ':' in start_str and len(end_str)==5 and ':' in end_str:
                start_time = datetime.time(int(start_str[0:2]), int(start_str[3:5]))
                end_time = datetime.time(int(end_str[0:2]), int(end_str[3:5]))
        # Format 2: 'HH:MM-HH:MM' (sometimes inside a list like ['HH:MM-HH:MM'])
        elif isinstance(slot_string, str) and '-' in slot_string:
             parts = slot_string.split('-')
             if len(parts) == 2 and len(parts[0])==5 and ':' in parts[0] and len(parts[1])==5 and ':' in parts[1]:
                 start_str, end_str = parts
                 start_time = datetime.time(int(start_str[0:2]), int(start_str[3:5]))
                 end_time = datetime.time(int(end_str[0:2]), int(end_str[3:5]))
        elif isinstance(slot_string, list) and len(slot_string) == 1 and isinstance(slot_string[0], str) and '-' in slot_string[0]:
            parts = slot_string[0].split('-')
            if len(parts) == 2 and len(parts[0])==5 and ':' in parts[0] and len(parts[1])==5 and ':' in parts[1]:
                 start_str, end_str = parts
                 start_time = datetime.time(int(start_str[0:2]), int(start_str[3:5]))
                 end_time = datetime.time(int(end_str[0:2]), int(end_str[3:5]))

        if start_time is None or end_time is None:
            return False

        # Check logic (inclusive end time)
        if start_time <= end_time: # Same day
            return start_time <= check_time <= end_time
        else: # Cross-midnight
            return start_time <= check_time <= datetime.time(23, 59, 59) or datetime.time(0, 0) <= check_time <= end_time

    except (ValueError, TypeError, IndexError):
        return False # Error during parsing

# --- Main Categorization Function ---
def determine_nightlife_category_with_nodata(hours_entry):
    """
    Categorizes a venue's hours, using None for missing/invalid data:
    - None (No Data)
    - 'Not Late (Before 9 pm)' (Closes before 21:00)
    - 'Evening (9pm-11pm)' (Open at/after 21:00, closes AT or BEFORE 23:00)
    - 'Late (After 11pm)' (Open after 23:00, closes AFTER 23:00, or 'late'/'24/7')
    """
    # Define target times
    time_21 = datetime.time(21, 0)
    time_23 = datetime.time(23, 0)

    # 1. Handle missing/invalid input -> return None
    if pd.isna(hours_entry): return None
    try:
        hours_dict = ast.literal_eval(str(hours_entry))
        if not isinstance(hours_dict, dict) or not hours_dict: return None # None for empty or non-dicts
    except (ValueError, SyntaxError): return None # None for malformed strings

    is_evening = False # Flag if open >= 21:00 AND closes <= 23:00
    is_late = False    # Flag if open > 23:00 OR closes > 23:00 OR crosses midnight OR 'late'/'24/7'

    try:
        # 2. Iterate through all slots
        for day_slots in hours_dict.values():
            for slot in day_slots:

                # Check keywords first (highest priority)
                if isinstance(slot, str) and slot in ('late', '24/7'):
                    is_late = True
                    break # Found the highest category for this day

                # Try parsing time ranges
                start_time = None
                end_time = None
                # (Parsing logic for start_time, end_time - remains the same)
                # Format 1: ['HH:MM', 'HH:MM']
                if isinstance(slot, list) and len(slot) == 2:
                     start_str, end_str = slot
                     if len(start_str)==5 and ':' in start_str and len(end_str)==5 and ':' in end_str:
                        start_time = datetime.time(int(start_str[0:2]), int(start_str[3:5]))
                        end_time = datetime.time(int(end_str[0:2]), int(end_str[3:5]))
                # Format 2: 'HH:MM-HH:MM' (or within a list)
                elif isinstance(slot, str) and '-' in slot:
                     parts = slot.split('-')
                     if len(parts) == 2 and len(parts[0])==5 and ':' in parts[0] and len(parts[1])==5 and ':' in parts[1]:
                         start_str, end_str = parts
                         start_time = datetime.time(int(start_str[0:2]), int(start_str[3:5]))
                         end_time = datetime.time(int(end_str[0:2]), int(end_str[3:5]))
                elif isinstance(slot, list) and len(slot) == 1 and isinstance(slot[0], str) and '-' in slot[0]:
                    parts = slot[0].split('-')
                    if len(parts) == 2 and len(parts[0])==5 and ':' in parts[0] and len(parts[1])==5 and ':' in parts[1]:
                        start_str, end_str = parts
                        start_time = datetime.time(int(start_str[0:2]), int(start_str[3:5]))
                        end_time = datetime.time(int(end_str[0:2]), int(end_str[3:5]))

                # If times were parsed successfully:
                if start_time and end_time:
                    # Check for cross-midnight (Late category)
                    if end_time < start_time:
                        is_late = True
                        break # Found the highest category for this day

                    # Check if open AFTER 23:00 OR closes AFTER 23:00 (Late category)
                    if start_time > time_23 or end_time > time_23:
                         is_late = True
                         break # Found the highest category for this day

                    # Check if open at or after 21:00 AND closes AT or BEFORE 23:00 (Evening category)
                    # Condition: interval overlaps with [21:00, 23:00] AND end_time <= 23:00
                    if ((start_time <= time_21 and end_time >= time_21) or \
                       (start_time >= time_21)) and end_time <= time_23:
                        is_evening = True # Mark as potentially Evening

            if is_late: # If we found 'late' within this day, stop checking the day
                 break

        if is_late: # If we found 'late' in any day, stop checking all days
             pass

    except Exception:
        return None # Error during iteration -> No Data

    # 3. Return the final category
    if is_late:
        return 'Late (After 11pm)'
    elif is_evening:
        return 'Evening (9pm-11pm)'
    else:
        # If not late and not evening, but data was valid (not None at start), it's 'Not Late'
        return 'Not Late (Before 9 pm)'

# --- 1. Apply the function to create the category column ---
# Replace existing column or create a new one
venues['operating_hours_category'] = venues['opening_hours_dict'].apply(determine_nightlife_category_with_nodata)

# --- 2. Show the resulting distribution, including missing data ---
print("\n--- Summary of Categories (with 'No Data') ---")
# Use dropna=False to explicitly show the count of None/NaN values
print(venues['operating_hours_category'].value_counts(dropna=False))




--- Summary of Categories (with 'No Data') ---
operating_hours_category
None                      2779
Evening (9pm-11pm)        2173
Not Late (Before 9 pm)    2009
Late (After 11pm)         1074
Name: count, dtype: int64


Adding new column 'operating_hours_category' in DB

In [12]:
from sqlalchemy import text, create_engine
import io

# Prepare DataFrame with only ID and the new category 
# Ensure the DataFrame is named 'venues' and has the 'operating_hours_category' column
df_to_update = venues[['venue_id', 'operating_hours_category']].copy()

# Replace any potential None/NaN values generated by the function with a default
# We use 'No Data' to explicitly represent missing hours information,
# which is different from 'Not Late' (meaning it closes before 9 pm).
df_to_update['operating_hours_category'].fillna('No Data', inplace=True)

print(f"Prepared {len(df_to_update)} rows for updating the category...")

# Use SQLAlchemy to perform DB operations
temp_table_name = "temp_venue_op_hours_cat"

try:
    with engine.connect() as connection:
        # Add the new column to the main table (if it doesn't exist)
        print("Adding column 'operating_hours_category' to venues (if needed)...")
        connection.execute(text(f"""
            ALTER TABLE berlin_source_data.venues
            ADD COLUMN IF NOT EXISTS operating_hours_category VARCHAR(30);
        """))
        # Clear any old values 
        connection.execute(text(f"""
             UPDATE berlin_source_data.venues SET operating_hours_category = NULL;
        """))
        connection.commit()

        # Create a temporary table
        print(f"Creating temporary table {temp_table_name}...")
        connection.execute(text(f"""
            DROP TABLE IF EXISTS {temp_table_name};
            CREATE TEMP TABLE {temp_table_name} (
                venue_id VARCHAR(30) PRIMARY KEY,
                category VARCHAR(30)
            );
        """))
        connection.commit() # Commit DDL

        # Load data into the temporary table
        print("Loading data into temporary table...")
        buffer = io.StringIO()
        # Use TAB separator for safety with potential commas in data (though unlikely here)
        # header=False because temp table doesn't expect headers
        df_to_update.to_csv(buffer, index=False, header=False, sep='\t')
        buffer.seek(0)

        # Get the underlying DBAPI connection (psycopg2) for copy_expert
        raw_conn = connection.connection
        cursor = raw_conn.cursor()

        # Use COPY FROM STDIN for fast bulk loading
        cursor.copy_expert(f"COPY {temp_table_name} FROM STDIN WITH (FORMAT CSV, DELIMITER E'\\t', NULL '')", buffer)
        raw_conn.commit() # Commit the COPY operation
        cursor.close()
        print("Data loaded into temporary table.")

        # Update the main table using the temporary table
        print("Updating main table 'venues'...")
        update_sql = text(f"""
            UPDATE berlin_source_data.venues v
            SET operating_hours_category = tmp.category
            FROM {temp_table_name} tmp
            WHERE v.venue_id = tmp.venue_id;
        """)
        result = connection.execute(update_sql)
        connection.commit() # Commit the UPDATE operation
        print(f"Updated {result.rowcount} rows in 'berlin_source_data.venues'.")

        # Clean up (optional, temp table drops automatically)
        connection.execute(text(f"DROP TABLE IF EXISTS {temp_table_name};"))
        connection.commit()

    print("✅ Update of 'operating_hours_category' in 'venues' table completed.")

except Exception as e:
    print(f"❌ An error occurred: {e}")
    # Attempt to rollback if a transaction was active
    try:
        if 'connection' in locals() and connection.in_transaction():
            connection.rollback()
            print("Transaction rolled back.")
    except Exception as rb_e:
        print(f"Error during rollback attempt: {rb_e}")

Prepared 8035 rows for updating the category...
Adding column 'operating_hours_category' to venues (if needed)...


C:\Users\IvI\AppData\Local\Temp\ipykernel_15952\423087330.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_to_update['operating_hours_category'].fillna('No Data', inplace=True)


Creating temporary table temp_venue_op_hours_cat...
Loading data into temporary table...
Data loaded into temporary table.
Updating main table 'venues'...
Updated 8035 rows in 'berlin_source_data.venues'.
✅ Update of 'operating_hours_category' in 'venues' table completed.


Final check

In [13]:
# SQL query to count categories in the database
sql_query = """
    SELECT
        operating_hours_category,
        COUNT(*) AS count
    FROM
        berlin_source_data.venues
    GROUP BY
        operating_hours_category
    ORDER BY
        count DESC;
"""

print("Executing query to count categories in the database...")

try:
    with engine.connect() as connection:
        # Execute the query and read the result directly into a DataFrame
        db_counts_df = pd.read_sql(text(sql_query), connection)

    print("\n--- Category distribution in 'venues' table (from Database) ---")
    print(db_counts_df.to_string(index=False)) # .to_string() for nice output

except Exception as e:
    print(f"❌ An error occurred while executing the query: {e}")

Executing query to count categories in the database...

--- Category distribution in 'venues' table (from Database) ---
operating_hours_category  count
                 No Data   2779
      Evening (9pm-11pm)   2173
  Not Late (Before 9 pm)   2009
       Late (After 11pm)   1074
